In [2]:
import numpy as np
import pandas as pd
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from ta import add_all_ta_features
import ta
from advanced_ta import LorentzianClassification
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [3]:

prediction = {
    'NEUTRAL': 0,
    'BUY': 1,
    'SELL': 2
}

In [38]:
def Extreme_Spike(df):
    # Define constants
    MINOR_MIN_EXTREME_HEIGHT_ATRS = 2.0
    MAJOR_TO_MINOR_HEIGHT_RATIO = 2.5
    MINOR_MIN_EXTREME_WIDTH = 2
    MAJOR_MIN_EXTREME_WIDTH = 2
    RANGE_AVERAGING_PERIOD = 250
    # LINE_VALUE_DOWN = -1.0
    # LINE_VALUE_UP = 1.0
    LINE_MINOR = 'minor'
    LINE_MAJOR = 'major'
    LINE_SHADOW = 'shadow'
    LINE_STABLE = 'stable'
    NoRepaint = False
    LINE_VALUE_UP = 1.0
    LINE_VALUE_FLAT = 0.0
    LINE_VALUE_DOWN = -1.0
    # Initialize columns for signals and other intermediate calculations
    df['ATR'] = df['high'] - df['low']
    df['ATR_SMA'] = df['ATR'].rolling(window=RANGE_AVERAGING_PERIOD).mean()
    df['MinorMinExtremeHeight'] = df['ATR_SMA'] * MINOR_MIN_EXTREME_HEIGHT_ATRS
    df['MajorMinExtremeHeight'] = df['MinorMinExtremeHeight'] * MAJOR_TO_MINOR_HEIGHT_RATIO
    df['line1'] = 0.0
    df['line2'] = 0.0
    df['line3'] = 0.0
    df['line4'] = 0.0
    df['line5'] = 0.0

    # Initialize state variables
    minor_low_extreme_price = df['low'].iloc[0]
    minor_hi_extreme_price = df['high'].iloc[0]
    major_low_extreme_price = df['low'].iloc[0]
    major_hi_extreme_price = df['high'].iloc[0]
    minor_low_extreme_idx = 0
    minor_hi_extreme_idx = 0
    major_low_extreme_idx = 0
    major_hi_extreme_idx = 0
    minor_extreme_mode = 0
    major_extreme_mode = 0
    first_minor_low = True
    first_minor_high = True
    first_major_low = True
    first_major_high = True

    def eraseExtreme(lineType, barIdx, value):
        drawShadow = (lineType == LINE_MAJOR) and (NoRepaint or (value == LINE_VALUE_UP and df['line1'].iloc[barIdx] != 0) or (value == LINE_VALUE_DOWN and df['line2'].iloc[barIdx] != 0))
        drawExtreme(lineType, barIdx, LINE_VALUE_FLAT)
        if drawShadow:
            draw(LINE_SHADOW, barIdx, value)

    def drawExtreme(lineType, barIdx, value):
        if not NoRepaint:
            draw(lineType, barIdx, value)
            drawStableLine(lineType, barIdx, value)

    def drawStableLine(lineType, barIdx, value):
        if lineType == LINE_MAJOR:
            return False
        draw(LINE_STABLE, barIdx, value)
        return True

    def draw(lineType, barIdx, value):
        if lineType == LINE_MAJOR:
            updateLine('line1', 'line2', barIdx, value)
        elif lineType == LINE_MINOR:
            updateLine('line5', 'line5', barIdx, value)
        elif lineType == LINE_SHADOW:
            updateLine('line3', 'line3', barIdx, value)
        elif lineType == LINE_STABLE:
            updateLine('line4', 'line4', barIdx, value)

    def updateLine(lineUp, lineDown, barIdx, value):
        if value in [LINE_VALUE_FLAT, LINE_VALUE_UP]:
            df.loc[barIdx, lineUp] = value
        if value in [LINE_VALUE_FLAT, LINE_VALUE_DOWN]:
            df.loc[barIdx, lineDown] = value

    # Helper functions
    def check_for_extremes(low_extreme_idx, low_extreme_price, hi_extreme_idx, hi_extreme_price, first_low, first_high, extreme_mode, min_extreme_height, min_extreme_width, current_idx, low, high, lineType, df):
        signal = 0
        line_value = 0.0
        # Check for Bottom
        if extreme_mode > -1:
            if low < low_extreme_price:
                if not first_low:
                    eraseExtreme(lineType, low_extreme_idx, LINE_VALUE_DOWN)
                low_extreme_price = low
                low_extreme_idx = current_idx
                first_low = False
            elif low > low_extreme_price:
                drawExtreme(lineType, low_extreme_idx, LINE_VALUE_DOWN)
                first_low = False
                if ((low - low_extreme_price) >= min_extreme_height) and ((current_idx - low_extreme_idx) >= min_extreme_width):
                    extreme_mode = -1
                    hi_extreme_price = high
                    hi_extreme_idx = current_idx
                    first_high = True
                    first_low = True
                    line_value = LINE_VALUE_DOWN
                    if NoRepaint:
                        draw(lineType, low_extreme_idx, LINE_VALUE_DOWN)
                    drawStableLine(lineType, low_extreme_idx, LINE_VALUE_FLAT)
                    # if signal_type == 'minor':
                    #     signal = 1  # Minor buy signal
                    #     df.at[low_extreme_idx, 'line1'] = line_value
                    # else:
                    #     signal = 2  # Major buy signal
                    #     df.at[low_extreme_idx, 'line2'] = line_value

        # Check for Top
        if extreme_mode < 1:
            if high > hi_extreme_price:
                if not first_high:
                    eraseExtreme(lineType, hi_extreme_idx, LINE_VALUE_UP)
                hi_extreme_price = high
                hi_extreme_idx = current_idx
                first_high = False
            elif high < hi_extreme_price:
                drawExtreme(lineType, hi_extreme_idx, LINE_VALUE_UP)
                first_high = False
                if ((hi_extreme_price - low) >= min_extreme_height) and ((current_idx - hi_extreme_idx) >= min_extreme_width):
                    extreme_mode = 1
                    low_extreme_price = low
                    low_extreme_idx = current_idx
                    first_high = True
                    first_low = True
                    line_value = LINE_VALUE_UP
                    if NoRepaint:
                        draw(lineType, hi_extreme_idx, LINE_VALUE_UP)
                    drawStableLine(lineType, hi_extreme_idx, LINE_VALUE_FLAT)
                    # if signal_type == 'minor':
                    #     signal = -1  # Minor sell signal
                    #     df.at[hi_extreme_idx, 'line1'] = line_value
                    # else:
                    #     signal = -2  # Major sell signal
                    #     df.at[hi_extreme_idx, 'line2'] = line_value

        return low_extreme_idx, hi_extreme_idx, low_extreme_price, hi_extreme_price, first_low, first_high, extreme_mode, signal

    # Process each row
    for idx in range(1, len(df)):
        # Minor extremes

        minor_low_extreme_idx, minor_hi_extreme_idx, minor_low_extreme_price, minor_hi_extreme_price, first_minor_low, first_minor_high, minor_extreme_mode, minor_signal = check_for_extremes(
            minor_low_extreme_idx, minor_low_extreme_price,
            minor_hi_extreme_idx, minor_hi_extreme_price, first_minor_low, first_minor_high, minor_extreme_mode,
            df['MinorMinExtremeHeight'].iloc[idx], MINOR_MIN_EXTREME_WIDTH, idx, df['low'].iloc[idx], df['high'].iloc[idx], 'minor', df
        )

        # Major extremes

        major_low_extreme_idx, major_hi_extreme_idx, major_low_extreme_price, major_hi_extreme_price, first_major_low, first_major_high, major_extreme_mode, major_signal = check_for_extremes(
            major_low_extreme_idx, major_low_extreme_price,
            major_hi_extreme_idx, major_hi_extreme_price, first_major_low, first_major_high, major_extreme_mode,
            df['MajorMinExtremeHeight'].iloc[idx], MAJOR_MIN_EXTREME_WIDTH, idx, df['low'].iloc[idx], df['high'].iloc[idx], 'major', df
        )

    return df[['line1','line2','line4','line5']]

In [5]:
# import pandas as pd
# import numpy as np
# import ta_py
# # Load the CSV file
# df = pd.read_csv('common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing.csv', parse_dates=['datetime'])
# highclose = list(map(list,zip(df['high'],df['low'])))
# print(highclose)
# df['zigzag'] = pd.Series(ta_py.zigzag(highclose))
# tolerance = 1e-8

# # Generate zigzag signals based on the zigzag values
# df['zigzag_signal'] = df.apply(
#     lambda x: 1 if np.abs(x['zigzag'] - x['high']) < tolerance else 
#               2 if np.abs(x['zigzag'] - x['low']) < tolerance else 0, 
#     axis=1
# )

# # Drop rows where zigzag is NaN
# df.dropna(subset=['zigzag'], inplace=True)

# # Print the last few rows of the DataFrame
# print(df.tail())
# df.to_csv('common/MachineLearningModel/output/yourfile_with_zigzag.csv')


In [39]:
import pandas
def calculate_1(pd: pd.DataFrame, predict=True):
    # Add indicators to new DataFrame
    pd2 = pd.iloc[:, :7].copy(deep=True)
    pd2 = pd2+Extreme_Spike(pd2)
    pd2['next_close'] = pd2['close'].shift(-3)
    pd2['Prediction'] = np.where(pd2['open'] < pd2['next_close'], 1,
                                 np.where(pd2['open'] > pd2['next_close'], 2, 0)).astype('int32')
    # if predict:
    #     # Select rows where either Signal or MACDSignal is 1, Prediction is 1, and neither Signal nor MACDSignal is 2
    pd21 = pd2[
        ((pd2['line1'] == 1) | (pd2['line2'] == 1) | (pd2['line4'] == 1) | (pd2['line5'] == 1) )  & 
        (pd2['Prediction'] == 1)
    ]
    pd22 = pd2[
        ((pd2['line1'] == 2) | (pd2['line2'] == 2) | (pd2['line4'] == 2) | (pd2['line5'] == 2) )   & 
        (pd2['Prediction'] == 2)
    ]

    #     # Select rows where either Signal or MACDSignal is 2, Prediction is 2, and neither Signal nor MACDSignal is 1
    #     pd22 = pd2[
    #         ((pd2['Signal'] == 2) | (pd2['MACDSignal'] == 2)) & 
    #         (pd2['Prediction'] == 2) & 
    #         ((pd2['Signal'] != 1) & (pd2['MACDSignal'] != 1))
    #     ]
    #     pd23 = pd2[(pd2['Signal'] == 0) & (pd2['MACDSignal'] == 0) & (pd2['Prediction'] == 0) ]
    pd2 = pandas.concat([pd21,pd22])
    
    pd2.dropna(inplace=True)
    # Clean up
    # pd2.drop(columns=['sma_av3', 'sma_av6', 'Intersection', 'Macd_intersection'], inplace=True)
    # pd2.drop(columns=['sma_av3', 'sma_av6','macd','macd_signal','Intersection', 'Macd_intersection', 'next_close'], inplace=True)

    return pd2

In [40]:
def process_files(file_paths):
    pd_data = []
    for file_path in file_paths:
        df = pd.read_csv(file_path)
        cal = calculate_1(df)
        pd_data.append(cal)
    return pd.concat(pd_data)

first_list = ['EURUSD', 'EURCAD', 'EURJPY', 'EURGBP', 'EURAUD']
sc_list = ['EURUSD', 'EURCAD', 'EURJPY', 'EURGBP', 'USDCAD', 'USDJPY']
th_list = ['EURAUD', 'EURUSD', 'EURCAD', 'EURJPY', 'EURGBP', 'USDCAD', 'USDJPY']

file_paths = []

for curr in first_list:
    file_paths.append(f"common/MachineLearningModel/output/five_mins/{curr}_5_Min.csv")

for curr in sc_list:
    file_paths.append(f'common/MachineLearningModel/output/five_mins/{curr}_5_Min_1.csv')

for curr in th_list:
    file_paths.append(f'common/MachineLearningModel/output/five_mins/{curr}_5_Min_2.csv')

for curr in th_list:
    file_paths.append(f'common/MachineLearningModel/output/five_mins/{curr}_5_Min_3.csv')
for curr in th_list:
    file_paths.append(f'common/MachineLearningModel/output/five_mins/{curr}_5_Min_4.csv')

data = process_files(file_paths)
# data_1 = pd.concat(pd_data_1)

In [41]:
data.dropna(inplace=True)
data.reset_index(drop=True,inplace=True)
print(data.iloc[:,7:].head())
print(data.shape)


Empty DataFrame
Columns: [line1, line2, line3, line4, line5, low, open, symbol, volume, next_close, Prediction]
Index: []
(0, 18)


In [42]:


le = LabelEncoder()
le.fit_transform(data['Prediction'])
print(le.classes_)


[]


In [ ]:
# import pickle
# combine_final_model = pickle.dump(stacking_clf, open('combineclassifier.sav','wb'))

In [43]:
print(data['Prediction'].value_counts())
X = data[['line1', 'line2', 'line3', 'line4', 'line5']]
y = data.iloc[:, -1]
print(data.columns)
print(X.columns)
print(X.count())
# print(y.head())
X_train, X_test, y_train, y_test =train_test_split(
  X, y, test_size = 0.35,train_size=0.65, shuffle=False)



Series([], Name: count, dtype: int64)
Index(['ATR', 'ATR_SMA', 'MajorMinExtremeHeight', 'MinorMinExtremeHeight',
       'close', 'datetime', 'high', 'line1', 'line2', 'line3', 'line4',
       'line5', 'low', 'open', 'symbol', 'volume', 'next_close', 'Prediction'],
      dtype='object')
Index(['line1', 'line2', 'line3', 'line4', 'line5'], dtype='object')
line1    0
line2    0
line3    0
line4    0
line5    0
dtype: int64


ValueError: With n_samples=0, test_size=0.35 and train_size=0.65, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [ ]:
# Initialize XGBoost classifier
xgb_model = XGBClassifier(booster="gbtree",max_depth=14,min_child_weight = 2)
# Train the model
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
preds = xgb_model.predict(X_test)

# Evaluate the model
print(f"Accuracy on train data by XGBoost Classifier\
: {accuracy_score(y_train, xgb_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by XGBoost Classifier\
: {accuracy_score(y_test, preds)*100}")


In [ ]:

rf_model = RandomForestClassifier()
# Train the model
rf_model.fit(X_train, y_train)

# Make predictions on the test set  
preds = rf_model.predict(X_test)

# Evaluate the model
print(f"Accuracy on train data by RandomForest Classifier\
: {accuracy_score(y_train, rf_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by RandomForest Classifier\
: {accuracy_score(y_test, preds)*100}")

In [ ]:
final_rf_model = RandomForestClassifier()
final_rf_model.fit(X, y)

In [ ]:
final_xgb_model = XGBClassifier(booster="gbtree",max_depth=14,min_child_weight = 2)
final_xgb_model.fit(X, y)

In [ ]:
import pickle
# xgb_final_model = pickle.dump(final_xgb_model, open('common/ml_model/xgbclassifier_new_5.sav','wb'))
# rf_final_model = pickle.dump(final_rf_model, open('common/ml_model/rfclassifier_new_5.sav','wb'))